# Run the Wallstreet-AI Pipeline in Google Colab

This notebook sets up the `wallstreet-ai` project in Google Colab and runs its main analysis pipeline with a single query.

The pipeline does more than call an LLM. It parses the user query, selects the required analysis flow, gathers market/news data, builds the analysis context, and generates the final investment analysis response. The result is also saved to the configured JSONL log file.

How to use this notebook:
1. In the next code cell, set `LLM_MODEL_API_KEY` and review the default settings.
2. Run the setup cell to clone the repository and install dependencies.
3. Run the environment setup cell to write `.env`.
4. Optionally review the available personas.
5. In the query cell near the bottom, set `QUERY` and optionally `PERSONA_NAME`.
6. Run the pipeline cell and inspect the saved JSONL result.

This notebook will:
- Clone the repository if it is not already present
- Install the required packages
- Write a `.env` file and set environment variables
- Run `pipeline()`
- Print the analysis result and the latest saved log entry


## Step 1. Update your API key and default settings

In [1]:
from getpass import getpass

GITHUB_REPO_URL = "https://github.com/davidkim205/wallstreet-ai.git"

LLM_MODEL_API_KEY = getpass("OpenAI API key: ").strip()
LLM_MODEL_NAME = "gpt-5-mini"

# Output file names
LOG_FILE = "analysis_results.jsonl"
PERSONA_FILE = "persona.jsonl"

OpenAI API key: ··········


## Step 2. Clone the repository and install dependencies

Run the next cell as-is. It checks whether the repository already exists in Colab, clones it if needed, installs the required packages, and moves into the project directory.


In [3]:
import os
import subprocess
from pathlib import Path

REPO_DIR = Path("/content/wallstreet-ai")

if not LLM_MODEL_API_KEY:
    raise ValueError("Please set LLM_MODEL_API_KEY before running this notebook.")

if REPO_DIR.exists():
    print(f"Repository already exists: {REPO_DIR}")
else:
    result = subprocess.run(
        ["git", "clone", GITHUB_REPO_URL, str(REPO_DIR)],
        text=True,
        capture_output=True,
    )

    print("returncode:", result.returncode)
    print("stdout:", result.stdout)
    print("stderr:", result.stderr)

    result.check_returncode()

subprocess.run(["python", "-m", "pip", "install", "-r", str(REPO_DIR / "requirements.txt")], check=True)

os.chdir(REPO_DIR)
print("Current working directory:", Path.cwd())

returncode: 0
stdout: 
stderr: Cloning into '/content/wallstreet-ai'...

Current working directory: /content/wallstreet-ai


## Step 3. Write environment variables

The next cell stores the values you entered above in both the current Python session and a local `.env` file inside the repository.


In [4]:
os.environ["LLM_MODEL_API_KEY"] = LLM_MODEL_API_KEY
os.environ["LLM_MODEL_NAME"] = LLM_MODEL_NAME
os.environ["LOG_FILE"] = LOG_FILE
os.environ["PERSONA_FILE"] = PERSONA_FILE

print("Environment variables configured for the current session.")

Environment variables configured for the current session.


## Step 4. Review available personas

If you want the analysis to reflect a specific investing style, run the next cell and copy one of the printed names into `PERSONA_NAME` in the query cell below. If you prefer the default behavior, keep `PERSONA_NAME = None`.


In [5]:
from persona.persona_loader import load_personas

personas = load_personas()
print(f"Loaded personas: {len(personas)}")
for idx, persona in enumerate(personas, start=1):
    print(f"{idx}. {persona.name}")

Loaded personas: 5
1. J.P. 모건
2. 워런 버핏
3. 짐 로저스
4. 켄 그리핀
5. 레이 달리오


## Step 5. Set the query and optional persona

Edit the next cell with the question you want to analyze. If you want to apply a persona, copy one of the names printed above into `PERSONA_NAME`. Otherwise, keep `PERSONA_NAME = None`.


In [6]:
# Set the query to run and the persona name to use.
# If you do not want to use a persona, keep PERSONA_NAME = None.
QUERY = "Summarize Nvidia's recent earnings and key investment points."
# PERSONA_NAME = "J.P. 모건"
PERSONA_NAME = None

## Step 6. Run the pipeline

Run the next cell to execute the analysis for the `QUERY` you just set.


In [7]:
from pipeline import pipeline, print_result

result = pipeline(
    QUERY,
    persona_name=PERSONA_NAME,
    stream=True,
)

print_result(result)


Wallstreet-AI 분석 시작: Summarize Nvidia's recent earnings and key investment points.
[②] Tool Router → 선택된 도구: ['earnings', 'fundamentals', 'news', 'web_search']
[③] 데이터 수집 중 (ticker=NVDA, period=1y)...
    → 펀더멘털: 52개 지표
    → 뉴스: 8개 헤드라인


/usr/local/lib/python3.12/dist-packages/yfinance/scrapers/fundamentals.py:36: DeprecationWarning: 'Ticker.earnings' is deprecated as not available via API. Look for "Net Income" in Ticker.income_stmt.
  warnings.warn("'Ticker.earnings' is deprecated as not available via API. Look for \"Net Income\" in Ticker.income_stmt.", DeprecationWarning)


    → 분기 실적: 5개 / 연간 실적: 4개 / EPS 서프라이즈: 0개
[INFO] - Nvidia earnings 검색 중...


[INFO] 기사 수집 진행: 0/3 [00:00<?]WARNING:trafilatura.utils:Language detector not installed, skipping detection
[INFO] 기사 수집 진행: 1/3 [00:00<00:01]WARNING:trafilatura.utils:Language detector not installed, skipping detection
[INFO] 기사 수집 진행: 2/3 [00:02<00:01]WARNING:trafilatura.utils:Language detector not installed, skipping detection
[INFO] 기사 수집 진행: 3/3 [00:02<00:00]


[INFO] - Nvidia guidance 검색 중...


...

[INFO] - Nvidia AI demand 검색 중...


[INFO] 기사 수집 진행: 0/3 [00:00<?]WARNING:trafilatura.utils:Language detector not installed, skipping detection
[INFO] 기사 수집 진행: 1/3 [00:00<00:01]WARNING:trafilatura.utils:Language detector not installed, skipping detection
[INFO] 기사 수집 진행: 2/3 [00:01<00:00]WARNING:trafilatura.utils:Language detector not installed, skipping detection
[INFO] 기사 수집 진행: 3/3 [00:01<00:00]


[INFO] - Nvidia stock reaction 검색 중...


[INFO] 기사 수집 진행: 0/3 [00:00<?]WARNING:trafilatura.utils:Language detector not installed, skipping detection
[INFO] 기사 수집 진행: 1/3 [00:00<00:01]WARNING:trafilatura.utils:Language detector not installed, skipping detection
[INFO] 기사 수집 진행: 2/3 [00:01<00:00]WARNING:trafilatura.utils:Language detector not installed, skipping detection
[INFO] 기사 수집 진행: 3/3 [00:01<00:00]


[⑤] 스트리밍 응답 수신 중...
[⑤] LLM 분석 스트리밍 생성 중 (Responses API, 모델: gpt-5-mini)...
This report is information only and not investment advice.

Summary of sources and constraints
- Analysis uses only numbers and text provided in your dataset and the attached web-news excerpts. Where a specific figure is not in the dataset, I mark it N/A. I also explicitly include the web-search / news items’ consensus comparisons and guidance as requested.

1) 핵심 실적 요약 (Revenue / Operating income / Net income + YoY)
- Latest reported quarter (labelled in dataset: 2026Q1)
  - Revenue: 0.07 조원
  - Operating income: 0.04 조원
  - OPM: 65.0%
  - Revenue YoY: +73.2%
  - Net income (dataset quarterly table): N/A
  - Net income (company press release / news included in dataset): $43.0B for the quarter (reported as “almost doubled” from prior year)
  - GAAP EPS (quarter, press release): $1.76; non‑GAAP EPS: $1.62

Notes on units: the dataset contains both the 0.07 / 0.04 (조원) figures and press-release USD figures (e.g.,

In [8]:
import json

log_path = Path(os.environ["LOG_FILE"])
if log_path.exists():
    last_line = log_path.read_text(encoding="utf-8").strip().splitlines()[-1]
    print(json.dumps(json.loads(last_line), ensure_ascii=False, indent=2))
else:
    print(f"Log file not found: {log_path}")

{
  "timestamp": "2026-04-01 01:48:24",
  "query": "Summarize Nvidia's recent earnings and key investment points.",
  "ticker": "NVDA",
  "analysis_type": "earnings",
  "data_context": {
    "ticker": "NVDA",
    "price_data": {},
    "fundamentals": {
      "company_name": "NVIDIA Corporation",
      "symbol": "NVDA",
      "exchange": "NMS",
      "quote_type": "EQUITY",
      "currency": "USD",
      "sector": "Technology",
      "industry": "Semiconductors",
      "country": "United States",
      "city": "Santa Clara",
      "website": "https://www.nvidia.com",
      "full_time_employees": 42000,
      "market_cap_b": 4238.79,
      "enterprise_value_b": 4187.87,
      "shares_outstanding_b": 24.3,
      "float_shares_b": 23.32,
      "pe_ratio": 35.51935,
      "forward_pe": 15.688602,
      "pb_ratio": 26.946846,
      "ps_ratio": 19.629671,
      "ev_to_ebitda": 31.433,
      "trailing_eps": 4.91,
      "forward_eps": 11.11635,
      "roe": 1.01485,
      "roa": 0.51188,
      